# Feature Engineering & Preprocessing Pipeline

**Project Title:** Feature Engineering & Preprocessing Pipeline  
**Notebook Focus:** Reusable, Leak-Free Scikit-Learn Preprocessing Pipeline (`ColumnTransformer`, Imputation, Scaling, One-Hot Encoding, Target Encoding, Correlation Analysis, & Unseen Category Handling)  
**Dataset:** Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`)  

---  
## Key Pipeline Architecture Highlights
1. **Leak-Free Preprocessing:** All scalers, imputers, and encoders fit strictly on `X_train` and `y_train`.
2. **Stratified Splitting:** Preserves the 26.54% churn target ratio identically across training (80%) and testing (20%) folds.
3. **Modular Sub-Pipelines:** Integrates numerical median imputation + Z-score scaling, categorical imputation + One-Hot Encoding (`handle_unknown='ignore'`), and Target Encoding.
4. **Production Inference Resilience:** Automatically handles unseen categories without throwing errors or NaNs.


In [1]:
import os
import sys
import numpy as np
import pandas as pd

# Include src directory in module path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.append(project_root)

from src.transformers import (
    NumericalImputer, CategoricalImputer, StandardScalerTransformer,
    OneHotEncoderTransformer, TargetEncoderTransformer, Pipeline, ColumnTransformer
)
from src.pipeline_builder import build_preprocessing_pipeline
from src.correlation_analysis import analyze_numerical_correlations
from src.leakage_checks import run_leakage_and_integrity_checks

print('Environment initialized successfully. Pandas version:', pd.__version__)


Environment initialized successfully. Pandas version: 3.0.1


## Step 1: Data Loading, Feature Schema Inspection, & Missingness Audit
We load the dataset, identify feature data types, target labels, and inspect missing values.

In [2]:
data_path = os.path.join(project_root, 'data', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = pd.read_csv(data_path)

print('=== DATASET OVERVIEW ===')
print(f'Total Rows: {len(df)}, Total Columns: {len(df.columns)}')
print('\nTarget Distribution (Churn):')
print(df['Churn'].value_counts(normalize=True))

# Audit Blank TotalCharges
blank_tc = (df['TotalCharges'].astype(str).str.strip() == '').sum()
print(f'\nBlank TotalCharges Count: {blank_tc}')
print('Tenure of Blank TotalCharges Rows:', df[df['TotalCharges'].astype(str).str.strip() == '']['tenure'].tolist())


=== DATASET OVERVIEW ===
Total Rows: 7043, Total Columns: 21

Target Distribution (Churn):
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

Blank TotalCharges Count: 11
Tenure of Blank TotalCharges Rows: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## Step 2: Stratified Train/Test Data Splitting

### Splitting Strategy Justification
* **Stratified 80/20 Split:** Preserves target class proportions (26.54% churn) identically in training ($N=5,636$) and test ($N=1,407$) sets.
* **Why Not Temporal Split?** Dataset provides elapsed `tenure` but lacks snapshot calendar timestamps. Sorting by tenure would distort feature distributions across folds.


In [3]:
y = (df['Churn'] == 'Yes').astype(int).values
X = df.drop(columns=['customerID', 'Churn']).copy()

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in X.columns if c not in num_cols]

# Leak-free Stratified Split
np.random.seed(42)
indices = np.arange(len(df))
pos_idx = indices[y == 1]
neg_idx = indices[y == 0]

np.random.shuffle(pos_idx)
np.random.shuffle(neg_idx)

n_pos_test = int(len(pos_idx) * 0.2)
n_neg_test = int(len(neg_idx) * 0.2)

test_idx = np.concatenate([pos_idx[:n_pos_test], neg_idx[:n_neg_test]])
train_idx = np.concatenate([pos_idx[n_pos_test:], neg_idx[n_neg_test:]])

X_train, y_train = X.iloc[train_idx].copy(), y[train_idx]
X_test, y_test = X.iloc[test_idx].copy(), y[test_idx]

print(f'Train Fold Shape: {X_train.shape}, Target Churn Proportion: {y_train.mean():.4f}')
print(f'Test Fold Shape : {X_test.shape}, Target Churn Proportion: {y_test.mean():.4f}')


Train Fold Shape: (5636, 19), Target Churn Proportion: 0.2654
Test Fold Shape : (1407, 19), Target Churn Proportion: 0.2651


## Step 3: Scikit-Learn Pipeline & ColumnTransformer Construction

We construct independent sub-pipelines for numerical scaling and categorical encoding, unified inside a `ColumnTransformer`.

In [4]:
pipeline = build_preprocessing_pipeline(num_cols, cat_cols)

# Fit strictly on training fold
pipeline.fit(X_train, y_train)

# Transform train and test sets
X_train_trans = pipeline.transform(X_train)
X_test_trans = pipeline.transform(X_test)

print('Pipeline Fit & Transform Successful!')
print(f'Transformed X_train Shape: {X_train_trans.shape}')
print(f'Transformed X_test Shape : {X_test_trans.shape}')
print(f'Engineered Features Count: {len(pipeline.get_feature_names_out())}')


Pipeline Fit & Transform Successful!
Transformed X_train Shape: (5636, 46)
Transformed X_test Shape : (1407, 46)
Engineered Features Count: 46


## Step 4: Numerical Feature Correlation Analysis

We analyze Pearson ($r$) and Spearman ($ho$) correlation matrices to detect multi-collinearity.

In [5]:
corr_results = analyze_numerical_correlations(X_train, num_cols)
print('=== PEARSON CORRELATION MATRIX ===')
print(corr_results['pearson'])

print('\n=== SPEARMAN CORRELATION MATRIX ===')
print(corr_results['spearman'])

print('\nHighly Correlated Feature Pairs (|r| >= 0.70):')
for pair in corr_results['high_corr_pairs']:
    print(f"  {pair['feature_1']} <-> {pair['feature_2']}: Pearson r = {pair['pearson_r']:.4f}, Spearman rho = {pair['spearman_rho']:.4f}")


=== PEARSON CORRELATION MATRIX ===
                  tenure  MonthlyCharges  TotalCharges
tenure          1.000000        0.243475      0.825268
MonthlyCharges  0.243475        1.000000      0.649138
TotalCharges    0.825268        0.649138      1.000000

=== SPEARMAN CORRELATION MATRIX ===
                  tenure  MonthlyCharges  TotalCharges
tenure          1.000000        0.272656      0.887945
MonthlyCharges  0.272656        1.000000      0.636967
TotalCharges    0.887945        0.636967      1.000000

Highly Correlated Feature Pairs (|r| >= 0.70):
  tenure <-> TotalCharges: Pearson r = 0.8253, Spearman rho = 0.8879


## Step 5: Inference Reusability & Unseen Category Resilience Test

We test the pipeline on synthetic new customer records containing unknown category strings (e.g. `PaymentMethod = 'Crypto / Bitcoin'`).

In [6]:
sample_unseen = X_test.iloc[:3].copy()
sample_unseen['PaymentMethod'] = ['Crypto / Bitcoin', 'Apple Pay', 'PayPal']
sample_unseen['InternetService'] = ['Satellite', 'Quantum Fiber', 'Starlink']

unseen_trans = pipeline.transform(sample_unseen)
print('Unseen Category Inference Output Shape:', unseen_trans.shape)
print('NaN Count in Inference Output:', np.isnan(unseen_trans).sum())
print('Pipeline successfully transformed unseen categories with 0 errors!')


Unseen Category Inference Output Shape: (3, 46)
NaN Count in Inference Output: 0
Pipeline successfully transformed unseen categories with 0 errors!


## Step 6: Automated Data Leakage & Pipeline Integrity Assertions

We execute automated assertions validating zero data leakage.

In [7]:
checks = run_leakage_and_integrity_checks(pipeline, X_train, X_test, y_train, y_test, num_cols)
print('=== AUTOMATED LEAKAGE & INTEGRITY CHECK RESULTS ===')
for check_name, status in checks.items():
    print(f'  {check_name:<35}: {status}')


=== AUTOMATED LEAKAGE & INTEGRITY CHECK RESULTS ===
  no_nans_pass                       : True
  train_scaled_mean_zero_pass        : True
  test_scaled_mean_diff_pass         : True
  unseen_category_resilience_pass    : True
  train_transformed_shape            : (5636, 46)
  test_transformed_shape             : (1407, 46)


## Step 7: Summary of Pipeline Architecture & Dataset Limitations

### Pipeline Architecture Summary
1. **Leak-Free Guarantee:** Transformers are fitted strictly on `X_train`. Holdout test metrics remain unpolluted.
2. **ColumnTransformer Integration:** Integrates median imputation, Z-score scaling, and One-Hot Encoding into a single executable object producing 46 dense output features.
3. **Inference Resilience:** Handles unseen category strings gracefully (`handle_unknown='ignore'`).

### Dataset Limitations
* **Cross-Sectional Data:** Absence of historical temporal snapshots.
* **High Multi-Collinearity:** Strong linear association between `tenure` and `TotalCharges` ($r = 0.8253$).
